# 05 — Realism, No-Copying, and Diversity Evaluation

This notebook adapts your friend's `Realism_Diversity.ipynb` to your final KoVAE project structure.

It evaluates both final KoVAE synthetic methods:

```text
rollout_v1
posterior_bank_v2
```

Real data:

```text
data/processed/native_rates/
```

Synthetic data:

```text
data/synthetic_subjects/kovae/rollout_v1/
data/synthetic_subjects/kovae/posterior_bank_v2/
```

Outputs:

```text
results/realism_diversity/kovae/<method>/csv/
figures/realism_diversity/kovae/<method>/
results/realism_diversity/kovae/combined_*.csv
figures/realism_diversity/kovae/combined/
```

Metrics:

```text
Realism:
- absolute mean difference
- absolute standard deviation difference
- histogram overlap
- FFT log-magnitude MAE

No-copying:
- copy ratio
- near-duplicate rate

Diversity:
- synthetic diversity ratio
```

EDA and TEMP are evaluated separately.


In [ ]:

# ============================================================
# 05_realism_diversity_kovae.py
#
# Native-rate realism / no-copying / diversity evaluation
# adapted to the final KoVAE project structure.
#
# Evaluates:
#   - rollout_v1
#   - posterior_bank_v2
#
# Real data:
#   data/processed/native_rates/
#
# Synthetic data:
#   data/synthetic_subjects/kovae/<method>/
#
# Results:
#   results/realism_diversity/kovae/<method>/csv/
#   results/realism_diversity/kovae/combined_*.csv
#
# Figures:
#   figures/realism_diversity/kovae/<method>/
#   figures/realism_diversity/kovae/combined/
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple
import json
import math
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

EVAL_CONFIG = {
    "project_root": "/home/iailab42/khans1/projects/ir",

    "model_family": "kovae",

    "real_dir": "data/processed/native_rates",
    "synthetic_base_dir": "data/synthetic_subjects/kovae",

    "results_base_dir": "results/realism_diversity",
    "figures_base_dir": "figures/realism_diversity",

    "methods_to_evaluate": ["rollout_v1", "posterior_bank_v2"],

    "method_display_names": {
        "rollout_v1": "KoVAE-Rollout",
        "posterior_bank_v2": "KoVAE-Posterior",
    },

    # Same subject split used in training/generation.
    "train_subjects": ["S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"],
    "val_subjects": ["S14", "S15"],
    "test_subjects": ["S7", "S8", "S10"],

    # Realism compares synthetic with train by default because synthetic was generated
    # from a model trained on train subjects.
    # Options: "train", "test", "all"
    "realism_reference_split": "train",

    "activity_ids": [1, 2, 3, 4, 5, 6, 7, 8],

    "histogram_bins": 80,

    # Safe defaults. Use smaller values if runtime is slow.
    "max_real_windows_per_activity": 1000,
    "max_syn_windows_per_activity": 1000,
    "distance_batch_size": 256,

    "random_seed": 42,

    "save_plots": True,
    "show_plots": True,
}


# ============================================================
# Native-rate branch configs
# ============================================================

ARRAY_CONFIGS = {
    "ACC": {
        "real_filename": "all_X_acc_32hz.npy",
        "syn_filename": "generated_subjects_X_acc_32hz.npy",
        "hz": 32,
        "window_len": 256,
        "num_channels": 3,
        "channel_names": ["ACC_x", "ACC_y", "ACC_z"],
    },
    "BVP": {
        "real_filename": "all_X_bvp_64hz.npy",
        "syn_filename": "generated_subjects_X_bvp_64hz.npy",
        "hz": 64,
        "window_len": 512,
        "num_channels": 1,
        "channel_names": ["BVP"],
    },
    "SLOW": {
        "real_filename": "all_X_slow_4hz.npy",
        "syn_filename": "generated_subjects_X_slow_4hz.npy",
        "hz": 4,
        "window_len": 32,
        "num_channels": 2,
        "channel_names": ["EDA", "TEMP"],
    },
}


SIGNAL_CONFIGS = {
    "ACC_x": {"array_key": "ACC", "channel_idx": 0, "hz": 32, "window_len": 256},
    "ACC_y": {"array_key": "ACC", "channel_idx": 1, "hz": 32, "window_len": 256},
    "ACC_z": {"array_key": "ACC", "channel_idx": 2, "hz": 32, "window_len": 256},
    "BVP": {"array_key": "BVP", "channel_idx": 0, "hz": 64, "window_len": 512},
    "EDA": {"array_key": "SLOW", "channel_idx": 0, "hz": 4, "window_len": 32},
    "TEMP": {"array_key": "SLOW", "channel_idx": 1, "hz": 4, "window_len": 32},
}

SIGNAL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]


# ============================================================
# Basic helpers
# ============================================================

def set_random_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def require_file(path: Path) -> Path:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    return path


def make_dirs(*dirs: Path) -> None:
    for directory in dirs:
        directory.mkdir(parents=True, exist_ok=True)


def save_json(data: Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def method_display_name(method_name: str, config: Dict) -> str:
    return config.get("method_display_names", {}).get(method_name, method_name)


def subject_sort_key(value):
    text = str(value)

    if text.startswith("S") and text[1:].isdigit():
        return (0, int(text[1:]))

    digits = "".join(ch for ch in text if ch.isdigit())
    if digits:
        return (1, int(digits), text)

    return (2, text)


def safe_mean(values) -> float:
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.mean(values))


def safe_std(values) -> float:
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.std(values))


def safe_percentile(values, q: float) -> float:
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.percentile(values, q))


def get_base_paths(config: Dict) -> Dict[str, Path]:
    root = Path(config["project_root"])
    model_family = str(config["model_family"])

    return {
        "root": root,
        "real_dir": root / config["real_dir"],
        "synthetic_base_dir": root / config["synthetic_base_dir"],
        "results_model_dir": root / config["results_base_dir"] / model_family,
        "figures_model_dir": root / config["figures_base_dir"] / model_family,
        "configs_dir": root / "configs",
    }


def get_method_paths(base_paths: Dict[str, Path], method_name: str) -> Dict[str, Path]:
    results_dir = base_paths["results_model_dir"] / method_name
    figures_dir = base_paths["figures_model_dir"] / method_name

    return {
        "results_dir": results_dir,
        "csv_dir": results_dir / "csv",
        "figures_dir": figures_dir,
    }


def check_array_shape(X: np.ndarray, cfg: Dict, name: str) -> None:
    if X.ndim != 3:
        raise ValueError(f"{name}: expected [N,T,C], got {X.shape}")

    if X.shape[1] != int(cfg["window_len"]):
        raise ValueError(
            f"{name}: expected window length {cfg['window_len']}, got {X.shape[1]}"
        )

    if X.shape[2] != int(cfg["num_channels"]):
        raise ValueError(
            f"{name}: expected {cfg['num_channels']} channels, got {X.shape[2]}"
        )


# ============================================================
# Loading
# ============================================================

def load_real_common(real_dir: Path) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    y_path = require_file(real_dir / "all_y.npy")
    subject_path = require_file(real_dir / "all_subject.npy")
    metadata_path = require_file(real_dir / "all_metadata.csv")

    y = np.load(y_path).astype(np.int64)
    subjects = np.load(subject_path, allow_pickle=True).astype(str)
    metadata = pd.read_csv(metadata_path)

    if len(y) != len(subjects):
        raise ValueError(f"Real y/subject mismatch: {len(y)} vs {len(subjects)}")

    if len(metadata) != len(y):
        raise ValueError(f"Real metadata/y mismatch: {len(metadata)} vs {len(y)}")

    metadata = metadata.copy()
    metadata["subject"] = subjects
    metadata["activity_label"] = y
    metadata["array_index"] = np.arange(len(y), dtype=np.int64)

    return y, subjects, metadata


def load_synthetic_common(synthetic_dir: Path) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    y_path = require_file(synthetic_dir / "generated_subjects_all_y.npy")
    subject_path = require_file(synthetic_dir / "generated_subjects_all_subject.npy")
    metadata_path = require_file(synthetic_dir / "generated_subjects_metadata.csv")

    y = np.load(y_path).astype(np.int64)
    subjects = np.load(subject_path, allow_pickle=True).astype(str)
    metadata = pd.read_csv(metadata_path)

    if len(y) != len(subjects):
        raise ValueError(f"Synthetic y/subject mismatch: {len(y)} vs {len(subjects)}")

    if len(metadata) != len(y):
        raise ValueError(f"Synthetic metadata/y mismatch: {len(metadata)} vs {len(y)}")

    metadata = metadata.copy()
    metadata["synthetic_subject"] = subjects
    metadata["activity_label"] = y
    metadata["array_index"] = np.arange(len(y), dtype=np.int64)

    return y, subjects, metadata


def load_real_arrays(real_dir: Path) -> Dict[str, np.ndarray]:
    real_X = {}

    for array_key, cfg in ARRAY_CONFIGS.items():
        path = require_file(real_dir / cfg["real_filename"])
        X = np.load(path).astype(np.float32)

        check_array_shape(X, cfg, f"Real {array_key}")

        real_X[array_key] = X

        print(f"Loaded real {array_key}: {X.shape}")

    return real_X


def load_synthetic_arrays(synthetic_dir: Path) -> Dict[str, np.ndarray]:
    syn_X = {}

    for array_key, cfg in ARRAY_CONFIGS.items():
        path = require_file(synthetic_dir / cfg["syn_filename"])
        X = np.load(path).astype(np.float32)

        check_array_shape(X, cfg, f"Synthetic {array_key}")

        syn_X[array_key] = X

        print(f"Loaded synthetic {array_key}: {X.shape}")

    return syn_X


def apply_mask_to_data(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    metadata: pd.DataFrame,
    mask: np.ndarray,
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]:
    mask = np.asarray(mask, dtype=bool)

    X_out = {}

    for array_key, X in X_dict.items():
        X_out[array_key] = X[mask].astype(np.float32)

    y_out = y[mask].astype(np.int64)
    subjects_out = subjects[mask].astype(str)

    metadata_out = metadata.loc[mask].copy().reset_index(drop=True)
    metadata_out["array_index"] = np.arange(len(y_out), dtype=np.int64)

    return X_out, y_out, subjects_out, metadata_out


def filter_activities(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    metadata: pd.DataFrame,
    activity_ids: List[int],
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]:
    keep = np.isin(y, np.asarray(activity_ids, dtype=np.int64))

    return apply_mask_to_data(
        X_dict=X_dict,
        y=y,
        subjects=subjects,
        metadata=metadata,
        mask=keep,
    )


def filter_by_subjects(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    metadata: pd.DataFrame,
    selected_subjects: List[str],
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]:
    selected = set(str(s) for s in selected_subjects)
    keep = np.asarray([str(s) in selected for s in subjects], dtype=bool)

    return apply_mask_to_data(
        X_dict=X_dict,
        y=y,
        subjects=subjects,
        metadata=metadata,
        mask=keep,
    )


# ============================================================
# Metric helpers
# ============================================================

def get_signal_windows(
    X_dict: Dict[str, np.ndarray],
    signal_name: str,
) -> np.ndarray:
    cfg = SIGNAL_CONFIGS[signal_name]
    X = X_dict[cfg["array_key"]]
    channel_idx = int(cfg["channel_idx"])

    return X[:, :, channel_idx].astype(np.float32)


def sample_by_activity(
    X_signal: np.ndarray,
    y: np.ndarray,
    activity: int,
    max_windows: int,
    rng: np.random.Generator,
) -> np.ndarray:
    idx = np.where(y == int(activity))[0]

    if len(idx) == 0:
        return np.empty((0, X_signal.shape[1]), dtype=np.float32)

    if len(idx) > int(max_windows):
        idx = rng.choice(idx, size=int(max_windows), replace=False)

    return X_signal[idx].astype(np.float32)


def histogram_overlap(real_values: np.ndarray, syn_values: np.ndarray, bins: int) -> float:
    real_values = np.asarray(real_values, dtype=np.float64).reshape(-1)
    syn_values = np.asarray(syn_values, dtype=np.float64).reshape(-1)

    real_values = real_values[np.isfinite(real_values)]
    syn_values = syn_values[np.isfinite(syn_values)]

    if len(real_values) == 0 or len(syn_values) == 0:
        return np.nan

    combined = np.concatenate([real_values, syn_values])
    low = float(np.min(combined))
    high = float(np.max(combined))

    if not np.isfinite(low) or not np.isfinite(high) or low == high:
        return np.nan

    real_hist, bin_edges = np.histogram(real_values, bins=int(bins), range=(low, high), density=True)
    syn_hist, _ = np.histogram(syn_values, bins=bin_edges, density=True)

    bin_widths = np.diff(bin_edges)
    overlap = np.sum(np.minimum(real_hist, syn_hist) * bin_widths)

    return float(np.clip(overlap, 0.0, 1.0))


def fft_logmag_mae(real_windows: np.ndarray, syn_windows: np.ndarray) -> float:
    if len(real_windows) == 0 or len(syn_windows) == 0:
        return np.nan

    real_fft = np.fft.rfft(real_windows.astype(np.float64), axis=1)
    syn_fft = np.fft.rfft(syn_windows.astype(np.float64), axis=1)

    real_logmag = np.log1p(np.abs(real_fft))
    syn_logmag = np.log1p(np.abs(syn_fft))

    real_mean_spectrum = np.mean(real_logmag, axis=0)
    syn_mean_spectrum = np.mean(syn_logmag, axis=0)

    return float(np.mean(np.abs(real_mean_spectrum - syn_mean_spectrum)))


def flatten_windows(windows: np.ndarray) -> np.ndarray:
    if len(windows) == 0:
        return np.empty((0, 0), dtype=np.float32)

    return windows.reshape(len(windows), -1).astype(np.float32)


def nearest_mse_distances(
    A: np.ndarray,
    B: np.ndarray,
    batch_size: int,
    exclude_self: bool = False,
) -> np.ndarray:
    """
    Compute nearest-neighbor MSE from each row of A to rows of B.

    Uses batched squared distance:
        ||a-b||^2 = ||a||^2 + ||b||^2 - 2ab
    then divides by feature dimension to produce MSE.
    """
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)

    if len(A) == 0 or len(B) == 0:
        return np.array([], dtype=np.float32)

    if A.shape[1] != B.shape[1]:
        raise ValueError(f"Feature dimension mismatch: A={A.shape}, B={B.shape}")

    n_a = len(A)
    n_b = len(B)
    dim = int(A.shape[1])

    if exclude_self and n_a != n_b:
        raise ValueError("exclude_self=True requires A and B to have the same number of rows.")

    B_norm = np.sum(B * B, axis=1)[None, :]
    nearest = np.empty(n_a, dtype=np.float32)

    for start in range(0, n_a, int(batch_size)):
        end = min(start + int(batch_size), n_a)
        A_batch = A[start:end]

        A_norm = np.sum(A_batch * A_batch, axis=1)[:, None]
        distances = A_norm + B_norm - 2.0 * (A_batch @ B.T)
        distances = np.maximum(distances, 0.0) / float(dim)

        if exclude_self:
            rows = np.arange(start, end)
            cols = rows

            valid = (cols >= 0) & (cols < n_b)
            distances[np.arange(end - start)[valid], cols[valid]] = np.inf

        nearest[start:end] = np.min(distances, axis=1).astype(np.float32)

    return nearest


def summarize_mse(values: np.ndarray, prefix: str) -> Dict[str, float]:
    return {
        f"{prefix}_mean": safe_mean(values),
        f"{prefix}_std": safe_std(values),
        f"{prefix}_p01": safe_percentile(values, 1),
        f"{prefix}_p05": safe_percentile(values, 5),
        f"{prefix}_p50": safe_percentile(values, 50),
        f"{prefix}_p95": safe_percentile(values, 95),
        f"{prefix}_p99": safe_percentile(values, 99),
    }


# ============================================================
# Realism metrics
# ============================================================

def compute_realism_table(
    real_X_dict: Dict[str, np.ndarray],
    real_y: np.ndarray,
    syn_X_dict: Dict[str, np.ndarray],
    syn_y: np.ndarray,
    method_name: str,
    reference_name: str,
    config: Dict,
) -> pd.DataFrame:
    rng = np.random.default_rng(int(config["random_seed"]) + 101)

    rows = []

    for activity in config["activity_ids"]:
        print(f"Realism metrics | method={method_name} | activity={activity}")

        for signal_name in SIGNAL_NAMES:
            real_signal = get_signal_windows(real_X_dict, signal_name)
            syn_signal = get_signal_windows(syn_X_dict, signal_name)

            real_sample = sample_by_activity(
                X_signal=real_signal,
                y=real_y,
                activity=int(activity),
                max_windows=int(config["max_real_windows_per_activity"]),
                rng=rng,
            )

            syn_sample = sample_by_activity(
                X_signal=syn_signal,
                y=syn_y,
                activity=int(activity),
                max_windows=int(config["max_syn_windows_per_activity"]),
                rng=rng,
            )

            if len(real_sample) == 0 or len(syn_sample) == 0:
                warnings.warn(
                    f"Missing samples for method={method_name}, activity={activity}, signal={signal_name}"
                )
                continue

            real_values = real_sample.reshape(-1)
            syn_values = syn_sample.reshape(-1)

            real_mean = safe_mean(real_values)
            syn_mean = safe_mean(syn_values)
            real_std = safe_std(real_values)
            syn_std = safe_std(syn_values)

            signal_cfg = SIGNAL_CONFIGS[signal_name]

            rows.append(
                {
                    "method": method_name,
                    "method_display_name": method_display_name(method_name, config),
                    "activity": int(activity),
                    "signal": signal_name,
                    "source_array": signal_cfg["array_key"],
                    "channel_idx_in_source_array": int(signal_cfg["channel_idx"]),
                    "hz": int(signal_cfg["hz"]),
                    "window_len": int(signal_cfg["window_len"]),
                    "reference_split": reference_name,
                    "real_sampled_windows": int(len(real_sample)),
                    "synthetic_sampled_windows": int(len(syn_sample)),
                    "real_mean": real_mean,
                    "synthetic_mean": syn_mean,
                    "abs_mean_diff_lower_is_better": abs(real_mean - syn_mean),
                    "real_std": real_std,
                    "synthetic_std": syn_std,
                    "abs_std_diff_lower_is_better": abs(real_std - syn_std),
                    "histogram_overlap_0_to_1_higher_is_better": histogram_overlap(
                        real_values,
                        syn_values,
                        int(config["histogram_bins"]),
                    ),
                    "fft_logmag_mae_lower_is_better": fft_logmag_mae(real_sample, syn_sample),
                }
            )

    return pd.DataFrame(rows)


def save_realism_summary_tables(realism_df: pd.DataFrame, csv_dir: Path) -> Dict[str, pd.DataFrame]:
    csv_dir.mkdir(parents=True, exist_ok=True)

    by_activity_path = csv_dir / "realism_by_activity_signal.csv"
    realism_df.to_csv(by_activity_path, index=False)
    print("Saved:", by_activity_path)

    metric_cols = [
        "abs_mean_diff_lower_is_better",
        "abs_std_diff_lower_is_better",
        "histogram_overlap_0_to_1_higher_is_better",
        "fft_logmag_mae_lower_is_better",
    ]

    avg_by_signal = (
        realism_df
        .groupby(["method", "method_display_name", "signal"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    counts_by_signal = (
        realism_df
        .groupby(["method", "method_display_name", "signal"], as_index=False)[
            ["real_sampled_windows", "synthetic_sampled_windows"]
        ]
        .sum()
    )

    avg_by_signal = avg_by_signal.merge(
        counts_by_signal,
        on=["method", "method_display_name", "signal"],
        how="left",
    )

    avg_by_signal = avg_by_signal[
        ["method", "method_display_name", "signal", "real_sampled_windows", "synthetic_sampled_windows"]
        + metric_cols
    ]

    avg_by_signal_path = csv_dir / "realism_average_by_signal.csv"
    avg_by_signal.to_csv(avg_by_signal_path, index=False)
    print("Saved:", avg_by_signal_path)

    overall = (
        realism_df
        .groupby(["method", "method_display_name"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    overall_path = csv_dir / "realism_average_overall.csv"
    overall.to_csv(overall_path, index=False)
    print("Saved:", overall_path)

    return {
        "realism_by_activity_signal": realism_df,
        "realism_average_by_signal": avg_by_signal,
        "realism_average_overall": overall,
    }


# ============================================================
# Copying and diversity metrics
# ============================================================

def compute_copy_diversity_table(
    real_train_X_dict: Dict[str, np.ndarray],
    real_train_y: np.ndarray,
    syn_X_dict: Dict[str, np.ndarray],
    syn_y: np.ndarray,
    method_name: str,
    config: Dict,
) -> pd.DataFrame:
    rng = np.random.default_rng(int(config["random_seed"]) + 202)

    rows = []

    for activity in config["activity_ids"]:
        print(f"Copy/diversity metrics | method={method_name} | activity={activity}")

        for signal_name in SIGNAL_NAMES:
            real_signal = get_signal_windows(real_train_X_dict, signal_name)
            syn_signal = get_signal_windows(syn_X_dict, signal_name)

            real_sample = sample_by_activity(
                X_signal=real_signal,
                y=real_train_y,
                activity=int(activity),
                max_windows=int(config["max_real_windows_per_activity"]),
                rng=rng,
            )

            syn_sample = sample_by_activity(
                X_signal=syn_signal,
                y=syn_y,
                activity=int(activity),
                max_windows=int(config["max_syn_windows_per_activity"]),
                rng=rng,
            )

            if len(real_sample) < 2 or len(syn_sample) < 2:
                warnings.warn(
                    f"Not enough samples for method={method_name}, activity={activity}, signal={signal_name}"
                )
                continue

            real_flat = flatten_windows(real_sample)
            syn_flat = flatten_windows(syn_sample)

            real_to_real_nn = nearest_mse_distances(
                A=real_flat,
                B=real_flat,
                batch_size=int(config["distance_batch_size"]),
                exclude_self=True,
            )

            syn_to_train_real_nn = nearest_mse_distances(
                A=syn_flat,
                B=real_flat,
                batch_size=int(config["distance_batch_size"]),
                exclude_self=False,
            )

            syn_to_syn_nn = nearest_mse_distances(
                A=syn_flat,
                B=syn_flat,
                batch_size=int(config["distance_batch_size"]),
                exclude_self=True,
            )

            real_ref_mean = safe_mean(real_to_real_nn)
            syn_to_real_mean = safe_mean(syn_to_train_real_nn)
            syn_to_syn_mean = safe_mean(syn_to_syn_nn)
            real_ref_p01 = safe_percentile(real_to_real_nn, 1)

            if np.isfinite(real_ref_mean) and real_ref_mean > 1e-12:
                copy_ratio_mean = float(syn_to_real_mean / real_ref_mean)
                synthetic_diversity_ratio_mean = float(syn_to_syn_mean / real_ref_mean)
            else:
                copy_ratio_mean = np.nan
                synthetic_diversity_ratio_mean = np.nan

            if np.isfinite(real_ref_p01):
                near_duplicate_rate = float(np.mean(syn_to_train_real_nn <= real_ref_p01))
            else:
                near_duplicate_rate = np.nan

            signal_cfg = SIGNAL_CONFIGS[signal_name]

            row = {
                "method": method_name,
                "method_display_name": method_display_name(method_name, config),
                "activity": int(activity),
                "signal": signal_name,
                "source_array": signal_cfg["array_key"],
                "channel_idx_in_source_array": int(signal_cfg["channel_idx"]),
                "hz": int(signal_cfg["hz"]),
                "window_len": int(signal_cfg["window_len"]),
                "feature_dim": int(real_flat.shape[1]),
                "real_train_sampled_windows": int(len(real_sample)),
                "synthetic_sampled_windows": int(len(syn_sample)),
                **summarize_mse(real_to_real_nn, "real_train_to_nearest_real_train_mse"),
                **summarize_mse(syn_to_train_real_nn, "syn_to_nearest_train_real_mse"),
                **summarize_mse(syn_to_syn_nn, "syn_to_nearest_syn_mse"),
                "copy_ratio_mean": copy_ratio_mean,
                "near_duplicate_threshold_real_train_p01_mse": real_ref_p01,
                "near_duplicate_rate_p01_lower_is_better": near_duplicate_rate,
                "synthetic_diversity_ratio_mean": synthetic_diversity_ratio_mean,
            }

            rows.append(row)

    return pd.DataFrame(rows)


def save_copy_diversity_summary_tables(copy_df: pd.DataFrame, csv_dir: Path) -> Dict[str, pd.DataFrame]:
    csv_dir.mkdir(parents=True, exist_ok=True)

    by_activity_path = csv_dir / "copy_diversity_by_activity_signal.csv"
    copy_df.to_csv(by_activity_path, index=False)
    print("Saved:", by_activity_path)

    metric_cols = [
        "real_train_to_nearest_real_train_mse_mean",
        "syn_to_nearest_train_real_mse_mean",
        "syn_to_nearest_syn_mse_mean",
        "copy_ratio_mean",
        "near_duplicate_rate_p01_lower_is_better",
        "synthetic_diversity_ratio_mean",
    ]

    avg_by_signal = (
        copy_df
        .groupby(["method", "method_display_name", "signal"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    counts_by_signal = (
        copy_df
        .groupby(["method", "method_display_name", "signal"], as_index=False)[
            ["real_train_sampled_windows", "synthetic_sampled_windows"]
        ]
        .sum()
    )

    avg_by_signal = avg_by_signal.merge(
        counts_by_signal,
        on=["method", "method_display_name", "signal"],
        how="left",
    )

    avg_by_signal = avg_by_signal[
        ["method", "method_display_name", "signal", "real_train_sampled_windows", "synthetic_sampled_windows"]
        + metric_cols
    ]

    avg_by_signal_path = csv_dir / "copy_diversity_average_by_signal.csv"
    avg_by_signal.to_csv(avg_by_signal_path, index=False)
    print("Saved:", avg_by_signal_path)

    overall = (
        copy_df
        .groupby(["method", "method_display_name"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    overall_path = csv_dir / "copy_diversity_average_overall.csv"
    overall.to_csv(overall_path, index=False)
    print("Saved:", overall_path)

    return {
        "copy_diversity_by_activity_signal": copy_df,
        "copy_diversity_average_by_signal": avg_by_signal,
        "copy_diversity_average_overall": overall,
    }


# ============================================================
# Split summary
# ============================================================

def save_data_split_summary(
    real_train_y: np.ndarray,
    real_train_subjects: np.ndarray,
    real_val_y: np.ndarray,
    real_val_subjects: np.ndarray,
    real_test_y: np.ndarray,
    real_test_subjects: np.ndarray,
    syn_y: np.ndarray,
    syn_subjects: np.ndarray,
    csv_dir: Path,
    config: Dict,
) -> pd.DataFrame:
    rows = []

    split_items = [
        ("real_train", real_train_y, real_train_subjects),
        ("real_val", real_val_y, real_val_subjects),
        ("real_test", real_test_y, real_test_subjects),
        ("synthetic_all", syn_y, syn_subjects),
    ]

    for split_name, y, subjects in split_items:
        row = {
            "split": split_name,
            "num_windows": int(len(y)),
            "subjects": ",".join(sorted(np.unique(subjects.astype(str)), key=subject_sort_key)),
        }

        for activity in config["activity_ids"]:
            row[f"activity_{activity}_windows"] = int(np.sum(y == int(activity)))

        rows.append(row)

    df = pd.DataFrame(rows)

    out_path = csv_dir / "data_split_summary.csv"
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return df


# ============================================================
# Plotting
# ============================================================

def plot_grouped_metric(
    df: pd.DataFrame,
    metric_col: str,
    title: str,
    ylabel: str,
    save_path: Path,
    config: Dict,
) -> None:
    if df is None or len(df) == 0:
        return

    pivot = df.pivot(index="signal", columns="method_display_name", values=metric_col)
    pivot = pivot.reindex(SIGNAL_NAMES)

    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot(kind="bar", ax=ax)

    ax.set_title(title)
    ax.set_xlabel("Signal")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Method")
    fig.tight_layout()

    if bool(config["save_plots"]):
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)

    if bool(config["show_plots"]):
        plt.show()

    plt.close(fig)


def save_combined_plots(
    combined_realism_by_signal: pd.DataFrame,
    combined_copy_by_signal: pd.DataFrame,
    figures_dir: Path,
    config: Dict,
) -> None:
    figures_dir.mkdir(parents=True, exist_ok=True)

    plot_grouped_metric(
        df=combined_realism_by_signal,
        metric_col="histogram_overlap_0_to_1_higher_is_better",
        title="Histogram overlap by signal",
        ylabel="Overlap, higher is better",
        save_path=figures_dir / "combined_histogram_overlap_by_signal.png",
        config=config,
    )

    plot_grouped_metric(
        df=combined_realism_by_signal,
        metric_col="fft_logmag_mae_lower_is_better",
        title="FFT log-magnitude MAE by signal",
        ylabel="MAE, lower is better",
        save_path=figures_dir / "combined_fft_logmag_mae_by_signal.png",
        config=config,
    )

    plot_grouped_metric(
        df=combined_copy_by_signal,
        metric_col="copy_ratio_mean",
        title="Copy ratio by signal",
        ylabel="Ratio",
        save_path=figures_dir / "combined_copy_ratio_by_signal.png",
        config=config,
    )

    plot_grouped_metric(
        df=combined_copy_by_signal,
        metric_col="synthetic_diversity_ratio_mean",
        title="Synthetic diversity ratio by signal",
        ylabel="Ratio",
        save_path=figures_dir / "combined_synthetic_diversity_ratio_by_signal.png",
        config=config,
    )

    plot_grouped_metric(
        df=combined_copy_by_signal,
        metric_col="near_duplicate_rate_p01_lower_is_better",
        title="Near-duplicate rate by signal",
        ylabel="Rate, lower is better",
        save_path=figures_dir / "combined_near_duplicate_rate_by_signal.png",
        config=config,
    )


# ============================================================
# Method evaluation
# ============================================================

def prepare_real_splits(
    real_X_all: Dict[str, np.ndarray],
    real_y_all: np.ndarray,
    real_subjects_all: np.ndarray,
    real_meta_all: pd.DataFrame,
    config: Dict,
) -> Dict[str, Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]]:
    real_X_all, real_y_all, real_subjects_all, real_meta_all = filter_activities(
        X_dict=real_X_all,
        y=real_y_all,
        subjects=real_subjects_all,
        metadata=real_meta_all,
        activity_ids=config["activity_ids"],
    )

    train = filter_by_subjects(
        X_dict=real_X_all,
        y=real_y_all,
        subjects=real_subjects_all,
        metadata=real_meta_all,
        selected_subjects=config["train_subjects"],
    )

    val = filter_by_subjects(
        X_dict=real_X_all,
        y=real_y_all,
        subjects=real_subjects_all,
        metadata=real_meta_all,
        selected_subjects=config["val_subjects"],
    )

    test = filter_by_subjects(
        X_dict=real_X_all,
        y=real_y_all,
        subjects=real_subjects_all,
        metadata=real_meta_all,
        selected_subjects=config["test_subjects"],
    )

    all_data = (real_X_all, real_y_all, real_subjects_all, real_meta_all)

    return {
        "train": train,
        "val": val,
        "test": test,
        "all": all_data,
    }


def choose_realism_reference(
    real_splits: Dict[str, Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]],
    config: Dict,
) -> Tuple[Dict[str, np.ndarray], np.ndarray, str]:
    split_name = str(config["realism_reference_split"])

    if split_name not in {"train", "test", "all"}:
        raise ValueError("realism_reference_split must be 'train', 'test', or 'all'.")

    X, y, subjects, meta = real_splits[split_name]

    return X, y, f"real_{split_name}"


def evaluate_one_method(
    method_name: str,
    real_splits: Dict[str, Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]],
    base_paths: Dict[str, Path],
    config: Dict,
) -> Dict[str, pd.DataFrame]:
    method_paths = get_method_paths(base_paths, method_name)
    synthetic_dir = base_paths["synthetic_base_dir"] / method_name

    make_dirs(
        method_paths["results_dir"],
        method_paths["csv_dir"],
        method_paths["figures_dir"],
    )

    print("\n" + "#" * 100)
    print(f"Evaluating method: {method_name} ({method_display_name(method_name, config)})")
    print("#" * 100)
    print("Synthetic dir:", synthetic_dir)
    print("Results dir:  ", method_paths["results_dir"])
    print("Figures dir:  ", method_paths["figures_dir"])

    syn_y_all, syn_subjects_all, syn_meta_all = load_synthetic_common(synthetic_dir)
    syn_X_all = load_synthetic_arrays(synthetic_dir)

    syn_X_all, syn_y_all, syn_subjects_all, syn_meta_all = filter_activities(
        X_dict=syn_X_all,
        y=syn_y_all,
        subjects=syn_subjects_all,
        metadata=syn_meta_all,
        activity_ids=config["activity_ids"],
    )

    real_train_X, real_train_y, real_train_subjects, real_train_meta = real_splits["train"]
    real_val_X, real_val_y, real_val_subjects, real_val_meta = real_splits["val"]
    real_test_X, real_test_y, real_test_subjects, real_test_meta = real_splits["test"]

    split_summary = save_data_split_summary(
        real_train_y=real_train_y,
        real_train_subjects=real_train_subjects,
        real_val_y=real_val_y,
        real_val_subjects=real_val_subjects,
        real_test_y=real_test_y,
        real_test_subjects=real_test_subjects,
        syn_y=syn_y_all,
        syn_subjects=syn_subjects_all,
        csv_dir=method_paths["csv_dir"],
        config=config,
    )

    realism_real_X, realism_real_y, reference_name = choose_realism_reference(
        real_splits=real_splits,
        config=config,
    )

    print("\n" + "=" * 80)
    print("Computing realism metrics")
    print("=" * 80)

    realism_df = compute_realism_table(
        real_X_dict=realism_real_X,
        real_y=realism_real_y,
        syn_X_dict=syn_X_all,
        syn_y=syn_y_all,
        method_name=method_name,
        reference_name=reference_name,
        config=config,
    )

    realism_tables = save_realism_summary_tables(
        realism_df=realism_df,
        csv_dir=method_paths["csv_dir"],
    )

    print("\n" + "=" * 80)
    print("Computing copy/diversity metrics")
    print("=" * 80)

    copy_df = compute_copy_diversity_table(
        real_train_X_dict=real_train_X,
        real_train_y=real_train_y,
        syn_X_dict=syn_X_all,
        syn_y=syn_y_all,
        method_name=method_name,
        config=config,
    )

    copy_tables = save_copy_diversity_summary_tables(
        copy_df=copy_df,
        csv_dir=method_paths["csv_dir"],
    )

    method_summary = {
        "model_family": config["model_family"],
        "method": method_name,
        "method_display_name": method_display_name(method_name, config),
        "synthetic_dir": str(synthetic_dir),
        "results_dir": str(method_paths["results_dir"]),
        "figures_dir": str(method_paths["figures_dir"]),
        "csv_dir": str(method_paths["csv_dir"]),
        "realism_reference_split": config["realism_reference_split"],
        "max_real_windows_per_activity": int(config["max_real_windows_per_activity"]),
        "max_syn_windows_per_activity": int(config["max_syn_windows_per_activity"]),
        "distance_batch_size": int(config["distance_batch_size"]),
        "signals": SIGNAL_NAMES,
        "activity_ids": config["activity_ids"],
        "num_realism_rows": int(len(realism_df)),
        "num_copy_diversity_rows": int(len(copy_df)),
    }

    save_json(method_summary, method_paths["results_dir"] / "method_realism_diversity_summary.json")

    return {
        "split_summary": split_summary,
        **realism_tables,
        **copy_tables,
    }


# ============================================================
# Combined saving
# ============================================================

def save_combined_tables(
    method_results: Dict[str, Dict[str, pd.DataFrame]],
    base_paths: Dict[str, Path],
    config: Dict,
) -> Dict[str, Path]:
    results_dir = base_paths["results_model_dir"]
    figures_dir = base_paths["figures_model_dir"] / "combined"

    results_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    combined_realism = pd.concat(
        [tables["realism_by_activity_signal"] for tables in method_results.values()],
        ignore_index=True,
    )

    combined_realism_by_signal = pd.concat(
        [tables["realism_average_by_signal"] for tables in method_results.values()],
        ignore_index=True,
    )

    combined_realism_overall = pd.concat(
        [tables["realism_average_overall"] for tables in method_results.values()],
        ignore_index=True,
    )

    combined_copy = pd.concat(
        [tables["copy_diversity_by_activity_signal"] for tables in method_results.values()],
        ignore_index=True,
    )

    combined_copy_by_signal = pd.concat(
        [tables["copy_diversity_average_by_signal"] for tables in method_results.values()],
        ignore_index=True,
    )

    combined_copy_overall = pd.concat(
        [tables["copy_diversity_average_overall"] for tables in method_results.values()],
        ignore_index=True,
    )

    output_paths = {
        "combined_realism_by_activity_signal": results_dir / "combined_realism_by_activity_signal.csv",
        "combined_realism_average_by_signal": results_dir / "combined_realism_average_by_signal.csv",
        "combined_realism_average_overall": results_dir / "combined_realism_average_overall.csv",
        "combined_copy_diversity_by_activity_signal": results_dir / "combined_copy_diversity_by_activity_signal.csv",
        "combined_copy_diversity_average_by_signal": results_dir / "combined_copy_diversity_average_by_signal.csv",
        "combined_copy_diversity_average_overall": results_dir / "combined_copy_diversity_average_overall.csv",
    }

    combined_realism.to_csv(output_paths["combined_realism_by_activity_signal"], index=False)
    combined_realism_by_signal.to_csv(output_paths["combined_realism_average_by_signal"], index=False)
    combined_realism_overall.to_csv(output_paths["combined_realism_average_overall"], index=False)
    combined_copy.to_csv(output_paths["combined_copy_diversity_by_activity_signal"], index=False)
    combined_copy_by_signal.to_csv(output_paths["combined_copy_diversity_average_by_signal"], index=False)
    combined_copy_overall.to_csv(output_paths["combined_copy_diversity_average_overall"], index=False)

    for path in output_paths.values():
        print("Saved:", path)

    overall = combined_realism_overall.merge(
        combined_copy_overall,
        on=["method", "method_display_name"],
        how="outer",
    )

    overall_path = results_dir / "combined_method_comparison_overall.csv"
    overall.to_csv(overall_path, index=False)
    output_paths["combined_method_comparison_overall"] = overall_path
    print("Saved:", overall_path)

    save_combined_plots(
        combined_realism_by_signal=combined_realism_by_signal,
        combined_copy_by_signal=combined_copy_by_signal,
        figures_dir=figures_dir,
        config=config,
    )

    combined_summary = {
        "model_family": config["model_family"],
        "methods_evaluated": config["methods_to_evaluate"],
        "method_display_names": {
            method: method_display_name(method, config)
            for method in config["methods_to_evaluate"]
        },
        "realism_reference_split": config["realism_reference_split"],
        "signals": SIGNAL_NAMES,
        "activity_ids": config["activity_ids"],
        "results_model_dir": str(results_dir),
        "figures_model_dir": str(base_paths["figures_model_dir"]),
        "combined_outputs": {key: str(path) for key, path in output_paths.items()},
        "combined_figures_dir": str(figures_dir),
        "important_metrics": {
            "realism": [
                "abs_mean_diff_lower_is_better",
                "abs_std_diff_lower_is_better",
                "histogram_overlap_0_to_1_higher_is_better",
                "fft_logmag_mae_lower_is_better",
            ],
            "no_copying": [
                "copy_ratio_mean",
                "near_duplicate_rate_p01_lower_is_better",
            ],
            "diversity": [
                "synthetic_diversity_ratio_mean",
            ],
        },
    }

    summary_path = results_dir / "combined_realism_diversity_summary.json"
    save_json(combined_summary, summary_path)
    output_paths["combined_realism_diversity_summary"] = summary_path
    print("Saved:", summary_path)

    return output_paths


# ============================================================
# Main
# ============================================================

def main(config: Dict = EVAL_CONFIG) -> Dict[str, object]:
    set_random_seeds(int(config["random_seed"]))

    base_paths = get_base_paths(config)

    make_dirs(
        base_paths["results_model_dir"],
        base_paths["figures_model_dir"],
        base_paths["configs_dir"],
    )

    config_path = base_paths["configs_dir"] / f"realism_diversity_{config['model_family']}_config.json"
    save_json(config, config_path)

    print("=" * 100)
    print("Notebook 05: Realism / no-copying / diversity evaluation")
    print("=" * 100)
    print("Model family:", config["model_family"])
    print("Methods:", config["methods_to_evaluate"])
    print("Real dir:", base_paths["real_dir"])
    print("Synthetic base:", base_paths["synthetic_base_dir"])
    print("Results dir:", base_paths["results_model_dir"])
    print("Figures dir:", base_paths["figures_model_dir"])
    print("Config saved:", config_path)
    print("Signals:", SIGNAL_NAMES)

    print("\nLoading real native-rate data once...")
    real_y_all, real_subjects_all, real_meta_all = load_real_common(base_paths["real_dir"])
    real_X_all = load_real_arrays(base_paths["real_dir"])

    real_splits = prepare_real_splits(
        real_X_all=real_X_all,
        real_y_all=real_y_all,
        real_subjects_all=real_subjects_all,
        real_meta_all=real_meta_all,
        config=config,
    )

    print("\nReal split sizes:")
    for split_name, (_, y_split, subjects_split, _) in real_splits.items():
        print(
            f"  {split_name:>5}: windows={len(y_split):6d} | "
            f"subjects={sorted(np.unique(subjects_split.astype(str)), key=subject_sort_key)}"
        )

    method_results = {}

    for method_name in config["methods_to_evaluate"]:
        method_results[method_name] = evaluate_one_method(
            method_name=method_name,
            real_splits=real_splits,
            base_paths=base_paths,
            config=config,
        )

    combined_paths = save_combined_tables(
        method_results=method_results,
        base_paths=base_paths,
        config=config,
    )

    print("\n" + "=" * 100)
    print("Realism/diversity evaluation completed.")
    print("=" * 100)

    print("\nMain outputs:")
    print(" ", combined_paths["combined_method_comparison_overall"])
    print(" ", combined_paths["combined_realism_average_by_signal"])
    print(" ", combined_paths["combined_copy_diversity_average_by_signal"])
    print(" ", combined_paths["combined_realism_diversity_summary"])

    print("\nHow to interpret:")
    print("  histogram_overlap_0_to_1_higher_is_better: higher is better")
    print("  fft_logmag_mae_lower_is_better: lower is better")
    print("  copy_ratio_mean: values much below 1 may suggest copying risk")
    print("  near_duplicate_rate_p01_lower_is_better: lower is better")
    print("  synthetic_diversity_ratio_mean: near 1 is ideal; too low means low diversity")

    return {
        "base_paths": base_paths,
        "method_results": method_results,
        "combined_paths": combined_paths,
    }


if __name__ == "__main__":
    outputs = main(EVAL_CONFIG)


## Final run

The default final run evaluates:

```text
rollout_v1
posterior_bank_v2
```

with output foldering:

```text
results/realism_diversity/kovae/
figures/realism_diversity/kovae/
```

Runtime note:

If it is slow, reduce:

```python
EVAL_CONFIG["max_real_windows_per_activity"] = 500
EVAL_CONFIG["max_syn_windows_per_activity"] = 500
EVAL_CONFIG["distance_batch_size"] = 128
```


In [ ]:
EVAL_CONFIG["project_root"] = "/home/iailab42/khans1/projects/ir"
EVAL_CONFIG["model_family"] = "kovae"

EVAL_CONFIG["real_dir"] = "data/processed/native_rates"
EVAL_CONFIG["synthetic_base_dir"] = "data/synthetic_subjects/kovae"

EVAL_CONFIG["methods_to_evaluate"] = ["rollout_v1", "posterior_bank_v2"]
EVAL_CONFIG["method_display_names"] = {
    "rollout_v1": "KoVAE-Rollout",
    "posterior_bank_v2": "KoVAE-Posterior",
}

EVAL_CONFIG["realism_reference_split"] = "train"

EVAL_CONFIG["max_real_windows_per_activity"] = 1000
EVAL_CONFIG["max_syn_windows_per_activity"] = 1000
EVAL_CONFIG["distance_batch_size"] = 256

EVAL_CONFIG["save_plots"] = True
EVAL_CONFIG["show_plots"] = True

outputs = main(EVAL_CONFIG)
outputs["combined_paths"]
